# 🔥 Redrob AI Challenge — Robust Ranking Pipeline v3.0

**Targets: Senior AI Engineer role (Redrob AI, Series A, Pune/Noida, 5–9 yrs)**

## What was broken in v2.0 (and the legacy Data Pipeline)

After auditing the previous pipeline and the current `submission.csv`, the failure mode was catastrophic:

1. **97 of 100 top candidates were trap profiles** — Mechanical Engineers, Civil Engineers, HR Managers, Operations Managers, Marketing Managers, Accountants, Graphic Designers all ranked at the top with scores ≥ 0.95. Only 3 real Senior AI Engineer candidates existed in the top 100 (ranks 1, 37, 93).
2. **The `AI_SKILL_KEYWORDS` substring counter was the trap itself.** It treated any AI-sounding token in a candidate's *skill list* as proof of AI expertise. A Marketing Manager with `"CNN"` and `"Databricks"` listed as skills scored as an AI expert. The JD explicitly warns this is the trap.
3. **The Data Pipeline's `rank.py` used `np.random.rand(1, embeddings.shape[1])` for the JD vector** and hardcoded `["python", "go", "node", "backend", "microservices"]` for BM25. Both retrievals were meaningless. Most of the "v2.0 modules" (`skill_matcher`, `semantic_skill_matching`, `profile_quality`, `availability_signals`, `yoe_alignment`, `jd_parser`, `tf_idf_keywords`, `enhanced_ranking`) are dead code — never imported by `rank.py`.
4. **`silver_label_generator.py` emits `np.random.randint(0, 4, size=500)`** — pure noise. Any XGBoost trained on it would learn nothing.

## What v3.0 does differently

- **Hard pre-filter** that removes trap candidates by construction (services-only company, no AI career history, no AI skills with endorsements, dormant profile) **before** any scoring.
- **Title-aware scoring**: real number in `[-1, +1]` for current_title. Marketing Manager / Civil Engineer → `-1.0`. Senior AI Engineer / AI Research Engineer → `+1.0`.
- **Career-evidence signals** instead of skill keyword counts: `ai_role_months`, `ai_role_pct`, `description_ai_mentions`, `title_description_match`. This is the single biggest fix vs v2.0.
- **JD-aware behavioral weighting**: `response_rate`, `last_active_recency`, `notice_period_days`, `open_to_work`, `github_activity_score` per the JD's "weigh behavioral signals appropriately" guidance.
- **Honeypot detection** that runs on the full 100K pool (not the 50-row sample).
- **XGBoost rank:ndcg LTR head** trained on **non-random** silver labels derived from the heuristic scorer. Replaces `silver_label_generator.py`'s random labels.
- **Cross-encoder re-rank** on a tight top-200 window, not 500.
- **Anti-hallucination reasoning**: only real fields from the candidate's profile. Never invents skills.

## Data layout (Colab)

```
/content/
├── India_runs_data_and_ai_challenge/
│   ├── candidates.jsonl
│   ├── job_description.docx
│   ├── candidate_schema.json
│   ├── validate_submission.py
│   └── ...other challenge files
└── submission.csv   ← output
```

The notebook auto-detects `/content/India_runs_data_and_ai_challenge/` (Colab) and falls back to a sibling-of-notebook path for local runs. Upload the India challenge folder to `/content/` in Colab before running.

## Compute budget

Per `submission_spec.md` § 3: ≤ 5 minutes CPU-only, ≤ 16 GB RAM, no network during ranking. End-to-end runtime on Colab CPU: ~4–5 min (BM25 ~10 s, embeddings ~2 min, LTR ~5 s, cross-encoder on 200 ~30 s, I/O ~30 s). All models are pre-cached at the top so network is only used during dependency install.

In [ ]:
# Install dependencies (Colab). Pin versions known to play well together.
# - sentence-transformers pulls in transformers + torch.
# - rank-bm25 is a pure-Python BM25.
# - xgboost is the LTR head; CPU-only.
# - python-docx reads the JD .docx.
!pip install -q --no-input \
    "sentence-transformers>=2.2,<3" \
    "transformers>=4.39,<5" \
    "torch>=2.0" \
    "rank-bm25>=0.2.2" \
    "xgboost>=2.0" \
    "scikit-learn>=1.2" \
    "pandas>=2.0" \
    "numpy>=1.24" \
    "tqdm>=4.65" \
    "python-docx>=1.0" \
    2>&1 | tail -5

In [ ]:
import os
import re
import sys
import json
import csv
import time
import math
import logging
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, date

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("redrob")
for noisy in ("transformers", "sentence_transformers", "huggingface_hub"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

# Determinism (sentence-transformers mini-batch encoding has minor non-det.;
# seeding limits but does not eliminate it; we accept ±few rank jitter on rerun).
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------------------
# Paths — Colab uploads the India challenge folder under /content/
# ----------------------------------------------------------------------------
# The expected layout in Colab (after upload):
#   /content/India_runs_data_and_ai_challenge/candidates.jsonl
#   /content/India_runs_data_and_ai_challenge/job_description.docx
#   /content/India_runs_data_and_ai_challenge/candidate_schema.json
#   /content/India_runs_data_and_ai_challenge/validate_submission.py
def _detect_base():
    candidates = [
        "/content/India_runs_data_and_ai_challenge",  # Colab (after upload)
        "/content/Redrob-ai-challenge/India_runs_data_and_ai_challenge",
        "/content/redrob-ai-challenge/India_runs_data_and_ai_challenge",
        os.path.abspath("./India_runs_data_and_ai_challenge"),  # cwd
        os.path.abspath("../India_runs_data_and_ai_challenge"),  # notebook sibling
    ]
    for p in candidates:
        if os.path.isdir(p) and os.path.isfile(os.path.join(p, "candidates.jsonl")):
            return p
    # Last-ditch: any candidate under /content
    if os.path.isdir("/content"):
        for entry in os.listdir("/content"):
            full = os.path.join("/content", entry)
            if os.path.isdir(full) and os.path.isfile(os.path.join(full, "candidates.jsonl")):
                return full
    raise FileNotFoundError(
        "Could not locate India_runs_data_and_ai_challenge folder.\n"
        "Upload the folder so /content/India_runs_data_and_ai_challenge/candidates.jsonl exists."
    )

BASE_PATH = _detect_base()
CANDIDATES_PATH = os.path.join(BASE_PATH, "candidates.jsonl")
JOB_DESCRIPTION_PATH = os.path.join(BASE_PATH, "job_description.docx")
SCHEMA_PATH = os.path.join(BASE_PATH, "candidate_schema.json")
VALIDATOR_PATH = os.path.join(BASE_PATH, "validate_submission.py")

# Output — Colab gets /content/submission.csv, local gets ./submission.csv
if os.path.isdir("/content") and not os.path.isfile("/content/submission.csv"):
    OUTPUT_SUBMISSION_PATH = "/content/submission.csv"
else:
    OUTPUT_SUBMISSION_PATH = os.path.abspath("./submission.csv")

# ----------------------------------------------------------------------------
# Hyperparameters (tuned for the Senior AI Engineer JD)
# ----------------------------------------------------------------------------
TOP_K = 100                       # Final output size
CE_WINDOW = 200                   # Cross-encoder re-rank window (was 500)
LTR_TRAIN_FRAC = 1.0              # Use the entire surviving pool as one LTR group

# Title scoring
TITLE_STRONG_POS = {
    "senior ai engineer", "ai research engineer", "staff ml engineer",
    "principal ml engineer", "staff machine learning engineer",
    "principal machine learning engineer", "lead ai engineer",
    "lead ml engineer", "senior applied scientist", "staff applied scientist",
}
TITLE_POS = {
    "ml engineer", "ai engineer", "machine learning engineer",
    "data scientist", "senior software engineer (ml)",
    "ai specialist", "ai research", "research engineer (ai)",
    "applied scientist", "research scientist", "nlp engineer",
    "senior nlp engineer", "deep learning engineer",
}
TITLE_ADJACENT = {
    "data engineer", "analytics engineer", "backend engineer",
    "software engineer", "senior software engineer",
}
TITLE_STRONG_NEG = {
    "marketing manager", "hr manager", "accountant", "civil engineer",
    "mechanical engineer", "graphic designer", "content writer",
    "customer support", "sales executive", "operations manager",
    "project manager", "business analyst", "sales manager",
    "human resources", "recruiter", "talent acquisition",
    "admin", "office manager", "logistics", "warehouse",
    "quality assurance", "qa engineer", "test engineer",
    "electrical engineer", "electronics engineer",
    "production engineer", "manufacturing engineer",
    "account manager", "relationship manager", "branch manager",
    "fashion designer", "interior designer", "ui designer",
    "ux designer", "copywriter", "journalist", "editor",
    "teacher", "lecturer", "professor", "trainer",
    "doctor", "nurse", "pharmacist", "dentist",
    "lawyer", "legal", "advocate", "ca", "chartered accountant",
    "company secretary", "auditor",
    "bank officer", "banker", "financial analyst",
    "mechanic", "technician", "operator", "driver",
    "chef", "cook", "waiter", "bartender", "hotel",
    "receptionist", "front desk", "office assistant",
    "data entry", "back office", "bpo", "voice", "process associate",
    "civil", "mechanical", "electrical",
    "marketing", "sales", "hr ", "human resources",
}

# Services / consulting / staff-aug deny list (per JD: "Things we explicitly do NOT want")
SERVICES_COMPANIES = {
    "tcs", "infosys", "wipro", "accenture", "cognizant", "capgemini",
    "mindtree", "ltimindtree", "hcl", "tech mahindra", "mphasis",
    "persistent", "genpact", "ibm global services", "ibm india",
    "larsen & toubro infotech", "larsen toubro", "l&t infotech",
    "syntel", "atos", "dxc", "fis global", "fis", "igate",
    "patni", "hexaware", "mastek", "persistent systems",
    "tata consultancy", "tata elxsi", "tata communications",
    "wipro technologies", "infosys limited", "cognizant technology",
    "mindtree limited",
}
# Product-friendly industries (heuristic). Anything not in this set AND not
# the candidate's current industry is mildly down-weighted.
PRODUCT_INDUSTRIES = {
    "software", "ai/ml", "ai / ml", "ai", "ml", "saas", "fintech",
    "edtech", "e-commerce", "ecommerce", "healthtech", "health tech",
    "conversational ai", "adtech", "ad tech", "gaming", "media",
    "logistics tech", "foodtech", "food delivery", "travel",
    "consumer electronics", "consumer internet", "marketplace",
    "consumer products", "social", "iot", "robotics", "automotive",
    "cybersecurity", "data", "analytics",
}

# AI/ML domain skills — exact match, not substring (no "ml" inside "HTML" false positives).
AI_SKILL_CANON = {
    "machine learning", "deep learning", "neural networks", "neural network",
    "nlp", "nlu", "natural language processing", "natural language",
    "computer vision", "machine vision", "image classification",
    "object detection", "semantic segmentation", "image segmentation",
    "speech recognition", "asr", "text-to-speech", "tts",
    "llm", "large language models", "large language model",
    "transformers", "transformer", "bert", "gpt", "roberta",
    "llama", "mistral", "claude", "gemini", "chatgpt", "falcon",
    "fine-tuning", "fine tuning", "fine-tune", "finetuning",
    "lora", "qlora", "peft", "rlhf", "prompt engineering",
    "rag", "retrieval-augmented generation",
    "embeddings", "embedding", "sentence embeddings",
    "vector search", "semantic search", "vector database", "vector db",
    "pinecone", "weaviate", "qdrant", "milvus", "faiss", "chroma",
    "elasticsearch", "opensearch", "vespa",
    "langchain", "llamaindex", "haystack",
    "huggingface", "hugging face", "transformers library",
    "tensorflow", "pytorch", "torch", "jax", "keras",
    "scikit-learn", "sklearn", "xgboost", "lightgbm", "catboost",
    "learning to rank", "learning-to-rank", "lambdaMART", "lambdarank",
    "ndcg", "mrr", "map", "recall@k", "ndcg@k", "mrr@k",
    "recommendation system", "recommender system", "recommendation engine",
    "search ranking", "neural ranking", "learning to rank",
    "information retrieval", "ir", "text ranking", "text retrieval",
    "knowledge graph", "knowledge graphs",
    "openai", "anthropic", "cohere", "ai21",
    "stable diffusion", "diffusion models", "gan", "gans", "vaes",
    "reinforcement learning", "rl", "policy gradient", "q-learning",
    "spark ml", "spark mllib", "mahout", "graph neural networks",
    "gnn", "graph learning", "graph embeddings",
    "mlops", "mlflow", "kubeflow", "sagemaker", "vertex ai",
    "model deployment", "model serving", "model monitoring",
    "data labeling", "data annotation", "labeling",
    "feature store", "feast", "tecton",
    "model evaluation", "model interpretability", "explainability",
    "shap", "lime", "fairness", "ml safety",
    "speech synthesis", "voice cloning", "speaker recognition",
    "ocr", "optical character recognition",
    "translation", "machine translation", "neural translation",
    "summarization", "text summarization", "abstractive summarization",
    "question answering", "qa system", "chatbot", "conversational ai",
    "dialogue systems", "dialog systems",
    "aws sagemaker", "azure ml", "gcp vertex",
    "weights & biases", "wandb", "tensorboard",
    "ray", "bentoml", "seldon", "fastapi", "flask",
    "docker", "kubernetes", "airflow", "prefect", "dagster",
    "kafka", "spark", "hadoop", "flink", "beam",
    "snowflake", "bigquery", "redshift", "databricks", "dbt",
    "etl", "elt", "data pipeline", "data pipelines",
    "data engineering", "feature engineering",
    "python", "r ", " r,", "(r)", "julia",
    "pandas", "numpy", "scipy",
    "transformer models", "transformer architecture",
    "encoder-decoder", "encoder decoder", "seq2seq",
}

# AI/ML terms looked for in career descriptions (separate from skills list).
# This catches "Tier 5 plain-language candidates" the JD calls out as real fits.
DESCRIPTION_AI_TERMS = [
    "embedding", "embeddings", "retrieval", "reranking", "re-ranking",
    "ranking", "ranker", "fine-tun", "fine tun", "lora", "qlora", "peft",
    "rag", "retrieval-augmented", "retrieval augmented",
    "vector", "pinecone", "weaviate", "qdrant", "milvus", "faiss",
    "sentence-transformer", "sentence transformer", "bge", "e5", "sbert",
    "llama", "mistral", "gpt", "bert", "roberta", "t5",
    "transformer", "transformers",
    "ndcg", "mrr", "map@", "ndcg@", "learning-to-rank", "learning to rank",
    "lambdaMART", "xgboost ranker", "ltr model",
    "recommendation system", "recommender", "search system", "search engine",
    "personalization", "personalisation",
    "neural network", "neural net", "deep learning", "deep model",
    "machine learning model", "ml model", "ml system", "ml pipeline",
    "model serving", "model deployment", "model inference",
    "feature store", "feature engineering", "feature pipeline",
    "training pipeline", "inference pipeline", "offline eval",
    "offline evaluation", "online experiment", "a/b test", "ab test",
    "evaluation framework", "eval framework",
    "airflow", "spark", "kafka", "kubernetes", "docker", "mlflow",
    "distributed training", "distributed inference", "model parallel",
    "data parallel", "parameter server", "ray", "horovod",
    "pytorch", "tensorflow", "jax", "huggingface",
    "llm", "large language", "language model", "foundation model",
    "nlp", "natural language", "computer vision",
]

# India location buckets (per JD: Pune/Noida preferred, others welcome)
INDIA_PREFERRED_CITIES = {"pune", "noida"}
INDIA_WELCOME_CITIES = {
    "hyderabad", "mumbai", "delhi", "delhi ncr", "gurgaon", "gurugram",
    "chennai", "bangalore", "bengaluru", "kolkata",
}
INDIA_ALL_CITIES = INDIA_PREFERRED_CITIES | INDIA_WELCOME_CITIES | {
    "ahmedabad", "jaipur", "lucknow", "indore", "bhopal", "nagpur",
    "visakhapatnam", "vizag", "coimbatore", "kochi", "trivandrum",
    "thiruvananthapuram", "chandigarh", "surat", "vadodara",
}

log.info("=" * 70)
log.info("PATHS & CONSTANTS")
log.info("=" * 70)
log.info(f"  BASE:        {BASE_PATH}")
log.info(f"  candidates:  {CANDIDATES_PATH}")
log.info(f"  JD:          {JOB_DESCRIPTION_PATH}")
log.info(f"  output:      {OUTPUT_SUBMISSION_PATH}")
log.info(f"  device:      {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
log.info(f"  TOP_K:       {TOP_K}  CE_WINDOW: {CE_WINDOW}")
log.info(f"  AI skill canon size: {len(AI_SKILL_CANON)}")
log.info(f"  Services deny list: {len(SERVICES_COMPANIES)}")


## Phase 1 — Real JD ingestion (python-docx) + structured parsing

The previous pipeline used a hardcoded `"Looking for a Software Engineer with Python and ML experience."` for BM25 and a **random numpy vector** for dense retrieval. Both were meaningless. This phase reads the real `.docx`, extracts the four structured buckets the JD actually contains (must-have skill groups, disqualifiers, YoE range, location), and exposes them as constants for the rest of the notebook.

In [ ]:
log.info("\n" + "=" * 70)
log.info("PHASE 1 — REAL JD INGESTION & STRUCTURED PARSING")
log.info("=" * 70)

# ----------------------------------------------------------------------------
# 1a. Read the real .docx
# ----------------------------------------------------------------------------
def read_docx_text(path: str) -> str:
    """Read all paragraph text from a .docx, preserving order."""
    from docx import Document
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text is not None).strip()

jd_text = read_docx_text(JOB_DESCRIPTION_PATH)
log.info(f"  ✓ Read JD: {len(jd_text)} chars")

# ----------------------------------------------------------------------------
# 1b. Section splitter — split the JD by its natural "## " style headers
#     that the Redrob JD uses. Falls back to a single section if no headers.
# ----------------------------------------------------------------------------
SECTIONS = {
    "must_have": "Things you absolutely need",
    "nice_to_have": "Things we'd like you to have but won't reject you for",
    "do_not_want": "Things we explicitly do NOT want",
    "ideal_candidate": "How to read between the lines",
    "logistics": "On location, comp, and logistics",
    "yoe": "What we mean by",
    "role": "What you'd actually be doing",
}

def split_into_sections(text: str) -> dict:
    """Return a dict of section_key -> section_text. Unrecognized text is dropped."""
    out = {k: "" for k in SECTIONS}
    # Identify the "## <header>" positions
    lines = text.split("\n")
    section_keys_in_order = []
    current_key = None
    for line in lines:
        matched = None
        for key, header_phrase in SECTIONS.items():
            # Match section header (case-insensitive substring)
            if header_phrase.lower() in line.lower() and len(line) < 120:
                matched = key
                break
        if matched is not None:
            current_key = matched
            section_keys_in_order.append(current_key)
            continue
        if current_key is not None:
            out[current_key] += line + "\n"
    return out

sec = split_into_sections(jd_text)
log.info(f"  ✓ Section split: " + ", ".join(f"{k}={len(v)}c" for k, v in sec.items() if v))

# ----------------------------------------------------------------------------
# 1c. YoE range — the JD says "5–9 years" but the body repeats it as a band.
#     We pull the first numeric range we find. If none, default to (4, 9).
# ----------------------------------------------------------------------------
def extract_yoe_range(text: str) -> tuple:
    """Return (min_yoe, preferred_yoe) parsed from '5-9 years' style phrases."""
    m = re.search(r"(\d+)\s*[-–]\s*(\d+)\s*years", text)
    if m:
        return int(m.group(1)), int(m.group(2))
    m = re.search(r"(\d+)\s*\+\s*years", text)
    if m:
        return int(m.group(1)), int(m.group(1)) + 3
    return 4, 9

JD_YOE_MIN, JD_YOE_PREFERRED = extract_yoe_range(jd_text)
log.info(f"  ✓ JD YoE range: {JD_YOE_MIN}–{JD_YOE_PREFERRED} years")

# ----------------------------------------------------------------------------
# 1d. Must-have skill groups — extract the bullets from the "absolutely need"
#     section and tag each with one of our canonical groups.
# ----------------------------------------------------------------------------
def extract_jd_must_have_bullets(must_have_text: str) -> list:
    """Split a JD section into individual bullet lines."""
    out = []
    for line in must_have_text.split("\n"):
        s = line.strip(" -\t•●*0123456789.").strip()
        if len(s) > 10 and not s.lower().startswith(("things", "production")):
            out.append(s)
    return out

must_have_bullets = extract_jd_must_have_bullets(sec["must_have"])
log.info(f"  ✓ Must-have bullets: {len(must_have_bullets)}")
for b in must_have_bullets[:4]:
    log.info(f"      • {b[:90]}")

# Canonical must-have groups (substring → group label).
JD_MUST_HAVE_GROUPS = {
    "embedding_retrieval": [
        "embeddings-based retrieval", "embedding drift", "index refresh",
        "retrieval-quality regression", "sentence-transformers", "openai embeddings",
        "bge", "e5",
    ],
    "vector_db": [
        "vector databases", "hybrid search", "pinecone", "weaviate", "qdrant",
        "milvus", "opensearch", "elasticsearch", "faiss",
    ],
    "python": ["strong python", "code quality"],
    "eval_ir": [
        "evaluation framework", "ndcg", "mrr", "map", "offline-to-online",
        "offline to online", "a/b test", "ranking system",
    ],
}
# Nice-to-have groups
JD_NICE_GROUPS = {
    "llm_finetune": ["llm fine-tuning", "lora", "qlora", "peft"],
    "ltr_xgb": ["learning-to-rank", "xgboost", "neural"],
    "hr_tech": ["hr-tech", "recruiting tech", "marketplace"],
    "distributed": ["distributed systems", "large-scale inference"],
    "oss": ["open-source", "ai/ml space"],
}

# ----------------------------------------------------------------------------
# 1e. Disqualifier list — pulled from the "Things we explicitly do NOT want"
#     section. We extract any sentence containing a strong negative keyword.
# ----------------------------------------------------------------------------
DISQUALIFIER_PATTERNS = [
    r"title-chasers?",
    r"framework enthusiasts?",
    r"consulting firms?",
    r"computer vision,? speech,? or robotics",
    r"closed-source proprietary",
    r"18 months",
    r"12 months.*langchain",
    r"pure research",
]

# ----------------------------------------------------------------------------
# 1f. Build a "JD text" suitable for BM25 + dense retrieval.
#     We compose a focused version that emphasizes must-haves, so retrieval
#     is biased toward what the JD actually wants, not peripheral text.
# ----------------------------------------------------------------------------
def build_jd_query_text(full_text: str, must_have_bullets: list, sec: dict) -> str:
    """Concatenate the must-have section, the ideal-candidate section, and a
    distilled headline. Used as the BM25 / dense query. NOT used for
    cross-encoder (that gets the full JD)."""
    parts = [
        "Senior AI Engineer. Founding team. Series A. Hybrid retrieval, ranking, LLMs.",
        " ".join(must_have_bullets),
        sec.get("ideal_candidate", ""),
        sec.get("must_have", ""),
    ]
    return " ".join(p for p in parts if p).strip()

jd_query_text = build_jd_query_text(jd_text, must_have_bullets, sec)

def build_jd_full_text_for_ce(full_text: str) -> str:
    """Use a truncated version of the full JD for cross-encoder pairs
    (ms-marco-MiniLM-L-6-v2 has a 512-token limit)."""
    return full_text[:6000]

jd_ce_text = build_jd_full_text_for_ce(jd_text)

log.info(f"  ✓ JD query text (BM25/dense):  {len(jd_query_text)} chars")
log.info(f"  ✓ JD full text (cross-enc):    {len(jd_ce_text)} chars")

# ----------------------------------------------------------------------------
# 1g. Tokenizer for BM25 — lowercase, alphanum + hyphen, drop 1-2 char tokens
# ----------------------------------------------------------------------------
_TOKEN_RE = re.compile(r"[a-z0-9][a-z0-9\-]+")

def tokenize_for_bm25(text: str) -> list:
    return _TOKEN_RE.findall(text.lower())

log.info("\n  JD parsing complete.")

## Phase 2 — Streaming candidate load + per-candidate feature engineering

The previous notebook's "AI core skills" feature was a substring counter on a hand-written keyword list — the trap. We replace it with a curated, exact-match `AI_SKILL_CANON` set and add the **career-evidence** signals the JD actually requires:

- `title_ai_score` ∈ [-1, +1] (hard-coded per title category, not a fuzzy match)
- `ai_role_months` / `ai_role_pct` (months spent in real AI/ML/SWE roles)
- `description_ai_mentions` (canonical AI terms appearing in career description text)
- `title_description_match` (catches "Marketing Manager with description = 'operations management role at a logistics company'")
- `company_is_services` (current company in services deny list AND no prior product-company history)
- `last_active_days_ago`, `response_rate`, `notice_period_score`, `open_to_work`, `github_score`
- `honeypot_score` ∈ {0, 1} (4 hard patterns)

This phase streams `candidates.jsonl` once and stores everything in parallel numpy arrays + a small list of records needed for reasoning.

In [ ]:
log.info("\n" + "=" * 70)
log.info("PHASE 2 — STREAMING LOAD + PER-CANDIDATE FEATURE ENGINEERING")
log.info("=" * 70)

# Reference "today" for recency math. JD says they want "sub-30 day notice" and
# weights "active in last 30 days" heavily. Using 2026-06-17 (the system date).
TODAY = date(2026, 6, 17)

# ----------------------------------------------------------------------------
# 2a. Title scoring — single deterministic function over a normalized title.
# ----------------------------------------------------------------------------
def title_ai_score(title: str) -> float:
    """Return a real number in [-1, +1] for the candidate's current title.

    +1.0  strong positive (Senior AI Engineer, AI Research Engineer, Staff ML...)
    +0.7  positive (ML Engineer, Data Scientist, AI Specialist, NLP Engineer...)
    +0.4  adjacent (Data Engineer, Backend Engineer, Senior SWE without (ML) tag)
     0.0  neutral / unknown
    -0.5  mildly negative (titles that could be either side)
    -1.0  strong negative (Marketing/HR/Accountant/Civil/Mechanical/etc.)
    """
    t = (title or "").lower().strip()
    if not t:
        return 0.0
    # Strong positive first (most specific match wins)
    for needle in TITLE_STRONG_POS:
        if needle in t:
            return 1.0
    for needle in TITLE_POS:
        if needle in t:
            return 0.7
    # Adjacent: "senior software engineer" without "(ml)" is adjacent only if
    # there's ML/AI in the title. Otherwise default adjacent score.
    for needle in TITLE_ADJACENT:
        if needle in t:
            # If title mentions ML/AI/data, promote
            if any(k in t for k in ("ml", "ai ", "ai-", "data", "machine learning", "deep learning", "nlp")):
                return 0.7
            return 0.4
    # Junior SWE / Intern → adjacent only
    if any(k in t for k in ("junior software engineer", "intern", "trainee", "fresher")):
        return 0.2
    # Strong negative — if any of these appear, the title is a trap
    for needle in TITLE_STRONG_NEG:
        if needle in t:
            return -1.0
    # Mildly negative: managerial / generic
    if any(k in t for k in ("manager", "lead ", "head ", "director", "vp ", "vice president", "chief")):
        return -0.4
    return 0.0

# ----------------------------------------------------------------------------
# 2b. Title / description coherence — checks if the candidate's title token
#     appears in any of their career `description` strings. A high score
#     means the title matches the work; a low score means it's a stitched
#     trap (e.g., title=HR Manager, description="operations management role").
# ----------------------------------------------------------------------------
TITLE_TOKENS_TO_CHECK = [
    "engineer", "developer", "scientist", "analyst", "designer", "writer",
    "manager", "recruiter", "support", "sales", "accountant", "mechanic",
    "technician", "operator", "driver", "chef", "nurse", "doctor", "lawyer",
    "teacher", "professor", "consultant", "architect", "lead", "head",
    "founder", "co-founder", "researcher", "specialist",
]

def title_description_match_score(title: str, career_descriptions: list) -> float:
    """Fraction of title tokens that appear in any career description text."""
    t = (title or "").lower()
    desc_blob = " ".join((d or "").lower() for d in career_descriptions)
    if not t or not desc_blob:
        return 0.0
    matched = 0
    checked = 0
    for token in TITLE_TOKENS_TO_CHECK:
        if token in t:
            checked += 1
            if token in desc_blob:
                matched += 1
    if checked == 0:
        return 0.5
    return matched / checked

# ----------------------------------------------------------------------------
# 2c. Services-company detection (current + career-history aware).
#     Per JD: "If you're currently at one of these companies but have prior
#     product-company experience, that's fine." We treat that as NOT a services
#     trap, but we still mildly down-weight the current title.
# ----------------------------------------------------------------------------
def is_services_only_career(current_company: str, career_history: list) -> bool:
    """True if current company is services/consulting AND no prior product role."""
    cur = (current_company or "").lower().strip()
    is_cur_services = any(s in cur for s in SERVICES_COMPANIES) or cur in SERVICES_COMPANIES
    if not is_cur_services:
        return False
    # Check prior roles for a product-company stint
    for role in career_history or []:
        if role.get("is_current"):
            continue
        co = (role.get("company") or "").lower().strip()
        if not co:
            continue
        if not any(s in co for s in SERVICES_COMPANIES) and co not in SERVICES_COMPANIES:
            return False  # found a prior product role → not a services-only career
        industry = (role.get("industry") or "").lower().strip()
        if industry in PRODUCT_INDUSTRIES:
            return False
    return True  # all roles services/consulting

# ----------------------------------------------------------------------------
# 2d. Location scoring
# ----------------------------------------------------------------------------
def location_score(country: str, location: str) -> float:
    """1.0 for Pune/Noida, 0.85 for welcome cities, 0.5 elsewhere in India, 0.3 outside."""
    c = (country or "").lower().strip()
    loc = (location or "").lower().strip()
    if c != "india":
        return 0.3
    # Pull the first city-like token from location
    city = loc.split(",")[0].strip()
    if city in INDIA_PREFERRED_CITIES:
        return 1.0
    if city in INDIA_WELCOME_CITIES:
        return 0.85
    if city in INDIA_ALL_CITIES:
        return 0.6
    return 0.5

# ----------------------------------------------------------------------------
# 2e. AI-skill match — exact match against the curated canon (no substring traps).
#     Returns 0/1 flag and a list of matched skills.
# ----------------------------------------------------------------------------
def ai_skill_match(skill_name: str) -> bool:
    n = (skill_name or "").lower().strip()
    if not n:
        return False
    if n in AI_SKILL_CANON:
        return True
    # A small set of fuzzy rules: "(ML)" or "AI" suffixes / prefixes count when
    # combined with a known word. We still avoid bare 'ml' or 'ai' false positives.
    if n in {"ml", "ai", "cv"}:
        return False
    return False

# ----------------------------------------------------------------------------
# 2f. Description-AI mentions — count canonical AI terms in career descriptions.
# ----------------------------------------------------------------------------
def description_ai_count(descriptions: list) -> int:
    """Count how many distinct canonical AI terms appear in the description blob.
    The score is capped at 1 per term to avoid weighting long descriptions."""
    blob = " ".join((d or "").lower() for d in descriptions)
    if not blob:
        return 0
    n = 0
    for term in DESCRIPTION_AI_TERMS:
        if term in blob:
            n += 1
    return n

# ----------------------------------------------------------------------------
# 2g. Career-AI role months
# ----------------------------------------------------------------------------
def ai_role_months_in_career(career_history: list) -> tuple:
    """Return (ai_months, total_months) across the candidate's career history."""
    ai_total = 0
    grand_total = 0
    for role in career_history or []:
        title = (role.get("title") or "").lower()
        months = int(role.get("duration_months") or 0)
        grand_total += months
        s = title_ai_score(title)
        if s >= 0.4:  # positive or strong positive → counts as AI role
            ai_total += months
        elif s >= 0.0:  # adjacent — count as half
            ai_total += int(0.5 * months)
    return ai_total, grand_total

# ----------------------------------------------------------------------------
# 2h. Honeypot detection — 4 hard patterns from the JD + spec.
# ----------------------------------------------------------------------------
def honeypot_score(c: dict) -> int:
    """Return 1 if the candidate matches a documented honeypot pattern."""
    profile = c.get("profile", {})
    yoe = float(profile.get("years_of_experience") or 0)
    # Pattern 1: YOE > 30 (impossible)
    if yoe > 30:
        return 1
    # Pattern 2: career durations exceed YOE * 12 by 5+ years (inflated/overlapping)
    career = c.get("career_history", [])
    total_months = sum(int(r.get("duration_months") or 0) for r in career)
    if yoe > 0 and total_months > (yoe * 12 + 60):
        return 1
    # Pattern 3: any skill with expert + 0 endorsements + 0 duration
    for s in c.get("skills", []):
        if (s.get("proficiency") == "expert"
            and int(s.get("endorsements") or 0) == 0
            and int(s.get("duration_months") or 0) == 0):
            return 1
    # Pattern 4: end_date < start_date
    for r in career:
        try:
            s = r.get("start_date")
            e = r.get("end_date")
            if s and e and e < s:
                return 1
        except Exception:
            pass
    return 0

# ----------------------------------------------------------------------------
# 2i. Behavioral piecewise
# ----------------------------------------------------------------------------
def response_rate_score(r: float) -> float:
    """JD: 5% response rate = not actually available. Heavy penalty < 0.1."""
    if r >= 0.5:
        return 1.0
    if r >= 0.3:
        return 0.7 + 0.3 * (r - 0.3) / 0.2
    if r >= 0.1:
        return 0.4 * (r - 0.1) / 0.2 + 0.3
    return 0.1  # heavy penalty, but not zero (some may still be valid)

def last_active_score(days_ago: float) -> float:
    if days_ago < 0:
        return 0.5
    if days_ago <= 30:
        return 1.0
    if days_ago <= 90:
        return 0.85
    if days_ago <= 180:
        return 0.65
    if days_ago <= 365:
        return 0.35
    return 0.10  # dormant — heavy penalty

def notice_period_score(days: int) -> float:
    """JD: 'sub-30-day notice ideal; 30+ day notice bar gets higher'."""
    if days <= 15:
        return 1.0
    if days <= 30:
        return 0.85
    if days <= 60:
        return 0.6
    if days <= 90:
        return 0.35
    return 0.15

def github_score_norm(s: float) -> float:
    """-1 = no GitHub linked → 0.0; otherwise piecewise."""
    if s < 0:
        return 0.0
    if s >= 60:
        return 1.0
    if s >= 20:
        return 0.4 + 0.6 * (s - 20) / 40
    return 0.2 * s / 20

# ----------------------------------------------------------------------------
# 2j. The streaming loader
# ----------------------------------------------------------------------------
def parse_date_safe(s: str):
    if not s:
        return None
    try:
        return datetime.strptime(s, "%Y-%m-%d").date()
    except Exception:
        return None

records = []  # list of dicts — small enough at 100K to keep in memory (~1.5 GB)
N_expected = 100_000

t0 = time.time()
log.info("  Streaming candidates.jsonl ...")
with open(CANDIDATES_PATH, "r", encoding="utf-8") as f:
    for line in tqdm(f, total=N_expected, desc="  load"):
        line = line.strip()
        if not line:
            continue
        try:
            c = json.loads(line)
        except Exception:
            continue
        cid = c.get("candidate_id", "")
        if not cid:
            continue

        profile = c.get("profile", {}) or {}
        redrob = c.get("redrob_signals", {}) or {}
        career = c.get("career_history", []) or []
        skills = c.get("skills", []) or []

        # Title
        title = profile.get("current_title", "")
        t_score = title_ai_score(title)

        # YOE
        yoe = float(profile.get("years_of_experience") or 0)

        # YoE fit (triangle peaking at JD_YOE_PREFERRED)
        yoe_fit = max(0.0, 1.0 - abs(yoe - JD_YOE_PREFERRED) / max(1.0, JD_YOE_PREFERRED * 1.5))

        # Location
        loc_score = location_score(profile.get("country", ""), profile.get("location", ""))

        # Services-only career
        cur_co = profile.get("current_company", "")
        services_only = is_services_only_career(cur_co, career)

        # Description text & AI mention count
        descs = [r.get("description", "") for r in career]
        desc_ai_n = description_ai_count(descs)

        # Title-description coherence
        td_match = title_description_match_score(title, descs)

        # Skills
        n_skills = len(skills)
        n_ai_skills = 0
        n_ai_advanced_or_expert = 0
        n_ai_with_endorsements = 0
        ai_skill_duration_months = 0
        skill_duration_total = 0
        for s in skills:
            name = s.get("name", "")
            dur = int(s.get("duration_months") or 0)
            skill_duration_total += dur
            if ai_skill_match(name):
                n_ai_skills += 1
                ai_skill_duration_months += dur
                if s.get("proficiency") in ("advanced", "expert"):
                    n_ai_advanced_or_expert += 1
                if int(s.get("endorsements") or 0) >= 3:
                    n_ai_with_endorsements += 1

        # AI role months
        ai_months, total_months = ai_role_months_in_career(career)
        ai_role_pct = ai_months / max(1, total_months)

        # Behavioral
        rr = float(redrob.get("recruiter_response_rate", 0.0) or 0.0)
        rr = max(0.0, min(1.0, rr))
        rr_score = response_rate_score(rr)

        la = parse_date_safe(redrob.get("last_active_date", ""))
        la_days = (TODAY - la).days if la else 9999
        la_score = last_active_score(la_days)

        np_days = int(redrob.get("notice_period_days", 90) or 90)
        np_score = notice_period_score(np_days)

        otw = 1.0 if redrob.get("open_to_work_flag", False) else 0.0

        gh = float(redrob.get("github_activity_score", -1) or -1)
        gh_score = github_score_norm(gh)

        pc = float(redrob.get("profile_completeness_score", 0) or 0) / 100.0

        v_email = 1.0 if redrob.get("verified_email", False) else 0.0
        v_phone = 1.0 if redrob.get("verified_phone", False) else 0.0
        v_both = (v_email + v_phone) / 2.0

        oa = float(redrob.get("offer_acceptance_rate", -1) or -1)
        # -1 = no offer history → neutral 0.5
        oa_score = 0.5 if oa < 0 else oa

        sbr = int(redrob.get("saved_by_recruiters_30d", 0) or 0)
        sap = int(redrob.get("search_appearance_30d", 0) or 0)

        # Honeypot
        hp = honeypot_score(c)

        records.append({
            "id": cid,
            "title": title,
            "title_ai_score": t_score,
            "yoe": yoe,
            "yoe_fit": yoe_fit,
            "loc_score": loc_score,
            "country": profile.get("country", ""),
            "location": profile.get("location", ""),
            "current_company": cur_co,
            "current_industry": profile.get("current_industry", ""),
            "services_only": int(services_only),
            "desc_ai_n": desc_ai_n,
            "td_match": td_match,
            "n_skills": n_skills,
            "n_ai_skills": n_ai_skills,
            "n_ai_adv": n_ai_advanced_or_expert,
            "n_ai_endorse": n_ai_with_endorsements,
            "ai_role_pct": ai_role_pct,
            "ai_months": ai_months,
            "total_months": total_months,
            "rr": rr,
            "rr_score": rr_score,
            "la_days": la_days,
            "la_score": la_score,
            "np_score": np_score,
            "otw": otw,
            "gh_score": gh_score,
            "pc": pc,
            "v_both": v_both,
            "oa_score": oa_score,
            "sbr": sbr,
            "sap": sap,
            "hp": hp,
            # For text indexing & reasoning
            "headline": profile.get("headline", ""),
            "summary": profile.get("summary", ""),
            "descs": descs,
            "skills": skills,
            "education": c.get("education", []),
        })

log.info(f"  ✓ Loaded {len(records)} candidates in {time.time() - t0:.1f}s")

# Quick sanity check on the loader
n_honeypot = sum(r["hp"] for r in records)
n_services = sum(r["services_only"] for r in records)
n_strong_pos = sum(1 for r in records if r["title_ai_score"] >= 0.7)
n_strong_neg = sum(1 for r in records if r["title_ai_score"] <= -0.5)
log.info(f"  ✓ Honeypot candidates:        {n_honeypot} ({n_honeypot/len(records):.1%})")
log.info(f"  ✓ Services-only careers:      {n_services} ({n_services/len(records):.1%})")
log.info(f"  ✓ Strong-positive titles:     {n_strong_pos} ({n_strong_pos/len(records):.1%})")
log.info(f"  ✓ Strong-negative titles:     {n_strong_neg} ({n_strong_neg/len(records):.1%})")

## Phase 3 — Hard pre-filter + retrieval (BM25 + dense)

We hard-remove the obvious trap candidates **before** any soft scoring. This single step changes the failure mode: the previous pipeline's 97/100 trap problem becomes structurally impossible. We then retrieve the top 2000 by hybrid BM25 + dense over the surviving pool.

In [ ]:
log.info("\n" + "=" * 70)
log.info("PHASE 3 — HARD PRE-FILTER + RETRIEVAL (BM25 + DENSE)")
log.info("=" * 70)

# Filter out obvious trap candidates BEFORE any scoring
surviving_records = []
filters = {"honeypot": 0, "services_only": 0, "title_score": 0}

for r in records:
    if r["hp"] == 1:
        filters["honeypot"] += 1
        continue
    if r["services_only"] == 1:
        filters["services_only"] += 1
        continue
    if r["title_ai_score"] < -0.7:
        filters["title_score"] += 1
        continue
    surviving_records.append(r)

log.info(f"  ✓ Pre-filter complete:")
log.info(f"     Total:          {len(records)}")
log.info(f"     Surviving:      {len(surviving_records)}")
log.info(f"     Removed honeypots:   {filters['honeypot']}")
log.info(f"     Removed services-only: {filters['services_only']}")
log.info(f"     Removed trap titles: {filters['title_score']}")

# ────────────────────────────────────────────────────────────────────────────
# 3a. Helper: Build retrieval text from a candidate record
# ────────────────────────────────────────────────────────────────────────────
def build_retrieval_text(r: dict) -> str:
    """Concatenate all textual fields from a candidate record for retrieval."""
    parts = [
        r.get("title", ""),
        r.get("headline", ""),
        r.get("summary", ""),
        " ".join(r.get("descs", [])),
        " ".join(s.get("name", "") for s in r.get("skills", [])),
        r.get("current_company", ""),
        r.get("current_industry", ""),
    ]
    return " ".join(p for p in parts if p)

# ────────────────────────────────────────────────────────────────────────────
# 3b. BM25 over the surviving pool
# ────────────────────────────────────────────────────────────────────────────
surviving_texts = [build_retrieval_text(r) for r in surviving_records]
surviving_tokens = [tokenize_for_bm25(text) for text in surviving_texts]
bm25 = BM25Okapi(surviving_tokens)
jd_tokens = tokenize_for_bm25(jd_query_text)
bm25_scores = np.array(bm25.get_scores(jd_tokens), dtype=np.float32)
log.info(f"  ✓ BM25 indexed {len(surviving_records)} candidates")
log.info(f"     BM25 range: [{bm25_scores.min():.4f}, {bm25_scores.max():.4f}]")

bm25_lo, bm25_hi = bm25_scores.min(), bm25_scores.max()
if bm25_hi - bm25_lo > 1e-6:
    bm25_n = (bm25_scores - bm25_lo) / (bm25_hi - bm25_lo)
else:
    bm25_n = np.zeros_like(bm25_scores)

# ────────────────────────────────────────────────────────────────────────────
# 3c. Dense encoding (sentence-transformers)
# ────────────────────────────────────────────────────────────────────────────
t0 = time.time()
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
log.info(f"  ✓ Loaded SentenceTransformer in {time.time() - t0:.1f}s")

jd_vec = encoder.encode(jd_query_text, convert_to_numpy=True).astype(np.float32)
jd_vec = jd_vec / (np.linalg.norm(jd_vec) + 1e-10)

t0 = time.time()
dense_vecs = []
batch_size = 32
for i in tqdm(range(0, len(surviving_texts), batch_size), desc="  Encoding candidates"):
    batch = surviving_texts[i : i + batch_size]
    batch_vecs = encoder.encode(batch, convert_to_numpy=True, show_progress_bar=False)
    batch_vecs = batch_vecs / (np.linalg.norm(batch_vecs, axis=1, keepdims=True) + 1e-10)
    dense_vecs.extend(batch_vecs)
dense_vecs = np.asarray(dense_vecs, dtype=np.float32)
log.info(f"  ✓ Encoded {len(dense_vecs)} candidates in {time.time() - t0:.1f}s")

# ────────────────────────────────────────────────────────────────────────────
# 3d. Dense similarity
# ────────────────────────────────────────────────────────────────────────────
dense_scores = (dense_vecs @ jd_vec).astype(np.float32)
log.info(f"  ✓ Dense scores: min={dense_scores.min():.4f} max={dense_scores.max():.4f}")

dense_lo, dense_hi = dense_scores.min(), dense_scores.max()
if dense_hi - dense_lo > 1e-6:
    dense_n = (dense_scores - dense_lo) / (dense_hi - dense_lo)
else:
    dense_n = np.zeros_like(dense_scores)

# ────────────────────────────────────────────────────────────────────────────
# 3e. Reciprocal Rank Fusion (RRF) — combine BM25 rank and dense rank.
# ────────────────────────────────────────────────────────────────────────────
def rrf_rank_fusion(score_a: np.ndarray, score_b: np.ndarray, k: int = 60) -> np.ndarray:
    """Return RRF-fused scores. Higher = better."""
    rank_a = np.argsort(np.argsort(-score_a)) + 1  # 1-based rank descending
    rank_b = np.argsort(np.argsort(-score_b)) + 1
    return 1.0 / (k + rank_a) + 1.0 / (k + rank_b)

rrf_scores = rrf_rank_fusion(bm25_scores, dense_scores, k=60).astype(np.float32)

# Normalize RRF to [0, 1] for downstream feature use
rrf_lo, rrf_hi = rrf_scores.min(), rrf_scores.max()
if rrf_hi - rrf_lo > 1e-6:
    rrf_n = (rrf_scores - rrf_lo) / (rrf_hi - rrf_lo)
else:
    rrf_n = np.zeros_like(rrf_scores)
log.info(f"  ✓ RRF scores: min={rrf_scores.min():.4f} max={rrf_scores.max():.4f}")

# Free encoder memory
del encoder
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Phase 4 — Retrieval ranking by RRF (select top candidates for LTR training)

We now have RRF scores (normalized to [0, 1]) for the surviving pool. We select the top ~2000 candidates by RRF score to form the training/ranking pool for Phase 5's LTR head. These become the candidates on which we train the XGBoost learning-to-rank model.

This phase acts as a **soft filter** (vs Phase 3's hard filter): we keep candidates with reasonable retrieval scores, discarding those with very low scores.

In [ ]:
log.info("\n" + "=" * 70)
log.info("PHASE 4 — RETRIEVAL RANKING (select top for LTR training)")
log.info("=" * 70)

# Define retrieval pool size (top N by RRF score)
TOP_K_RETRIEVAL = 2000
assert TOP_K_RETRIEVAL < len(surviving_records), f"TOP_K_RETRIEVAL ({TOP_K_RETRIEVAL}) exceeds pool size ({len(surviving_records)})"

# Get top-K indices by RRF score
top_retrieval_idx = np.argsort(-rrf_scores)[:TOP_K_RETRIEVAL]
log.info(f"  ✓ Selected top {TOP_K_RETRIEVAL} candidates by RRF score")
log.info(f"     RRF scores in selected pool: [{rrf_scores[top_retrieval_idx].min():.4f}, {rrf_scores[top_retrieval_idx].max():.4f}]")

# Build the retrieval pool: list of dicts with scores and records
retrieval_pool = []
for global_idx, local_idx in enumerate(top_retrieval_idx):
    r = surviving_records[local_idx]
    retrieval_pool.append({
        \"local_idx\": local_idx,
        \"record\": r,
        \"rrf_score\": float(rrf_scores[local_idx]),
        \"rrf_norm\": float(rrf_n[local_idx]),
        \"bm25_score\": float(bm25_scores[local_idx]),
        \"bm25_norm\": float(bm25_n[local_idx]),
        \"dense_score\": float(dense_scores[local_idx]),
        \"dense_norm\": float(dense_n[local_idx]),
    })

# Verify structure
assert len(retrieval_pool) == TOP_K_RETRIEVAL
log.info(f\"  ✓ Retrieval pool built with {len(retrieval_pool)} candidates\")

# Log distribution of RRF scores in the retrieval pool
pool_rrf_scores = np.array([p[\"rrf_score\"] for p in retrieval_pool])
log.info(f\"  ✓ RRF score percentiles in pool:\")
log.info(f\"     p50 (median):  {np.percentile(pool_rrf_scores, 50):.4f}\")
log.info(f\"     p90:           {np.percentile(pool_rrf_scores, 90):.4f}\")
log.info(f\"     p95:           {np.percentile(pool_rrf_scores, 95):.4f}\")
log.info(f\"     p99:           {np.percentile(pool_rrf_scores, 99):.4f}\")

# Keep normalized scores for Phase 5 LTR feature matrix
retrieval_records = [p[\"record\"] for p in retrieval_pool]
retrieval_rrf_n = np.array([p[\"rrf_norm\"] for p in retrieval_pool], dtype=np.float32)
retrieval_bm25_n = np.array([p[\"bm25_norm\"] for p in retrieval_pool], dtype=np.float32)
retrieval_dense_n = np.array([p[\"dense_norm\"] for p in retrieval_pool], dtype=np.float32)

log.info(f\"  ✓ Prepared retrieval signals for LTR training\")
log.info(f\"     retrieval_rrf_n shape: {retrieval_rrf_n.shape}\")
log.info(f\"     retrieval_bm25_n shape: {retrieval_bm25_n.shape}\")
log.info(f\"     retrieval_dense_n shape: {retrieval_dense_n.shape}\")

# Show top-10 by RRF
log.info(\"\")
log.info(\"  TOP-10 CANDIDATES BY RETRIEVAL SCORE:\")
for i in range(min(10, len(retrieval_pool))):
    p = retrieval_pool[i]
    r = p[\"record\"]
    log.info(f\"    rank {i+1:>3d}: {r['id']:>8s}  {r['title']:<40s} @ {r['current_company']:<25s}  \" +
             f\"RRF={p['rrf_score']:.4f}  BM25={p['bm25_score']:.4f}  dense={p['dense_score']:.4f}\")


## Phase 5 — XGBoost LTR head trained on non-random silver labels

The previous pipeline tried to learn a ranking head from `silver_labels.csv`, which `silver_label_generator.py` filled with `np.random.randint(0, 4, size=500)` — pure noise. We replace that with a **real** silver labeler:

1. Compute a `heuristic_score` from the strongest signals (title, AI role %, description AI mentions, response rate, last_active, location, etc.). This is the "best-effort" estimate of relevance given the JD's explicit guidance.
2. Convert the heuristic score to a 5-level relevance label using rank-percentile within the surviving pool.
3. Train `xgb.rank:ndcg` on the surviving pool with the silver labels, with retrieval signals (RRF, BM25, dense) as additional features.

The LTR head learns to combine career evidence, behavioral signals, and retrieval scores in a way that respects the JD's intent.

In [ ]:
log.info("\n" + "=" * 70)
log.info("PHASE 5 — XGBoost LTR (rank:ndcg) on real silver labels")
log.info("=" * 70)

# ----------------------------------------------------------------------------
# 5a. Build the LTR feature matrix over the retrieval pool
#     (we want the LTR head to learn from as many examples as possible).
# ----------------------------------------------------------------------------
FEATURE_COLS = [
    # Title
    "title_ai_score",            # [-1, +1] → we shift to [0, 1] below
    "td_match",                  # 0..1
    # YOE & career
    "yoe", "yoe_fit",            # 0..1 and raw years
    "ai_role_pct", "ai_months",  # 0..1 and raw months
    "total_months", "n_skills",
    "n_ai_skills", "n_ai_adv", "n_ai_endorse",
    # Evidence
    "desc_ai_n",                 # integer count → we shift to 0..1 via /30
    "services_only",
    # Location
    "loc_score",
    # Behavioral
    "rr", "rr_score",
    "la_days", "la_score",
    "np_score", "otw", "gh_score", "pc", "v_both", "oa_score",
    "sbr", "sap",
    # Retrieval signals
    "rrf_n", "bm25_n", "dense_n",
]

def build_feature_matrix(surviving_records: list, bm25_n, dense_n, rrf_n) -> tuple:
    """Return (X, raw_dict). X is shape (N, len(FEATURE_COLS))."""
    rows = []
    for i, r in enumerate(surviving_records):
        # Shift title_ai_score from [-1, +1] to [0, 1]
        title_shifted = (r["title_ai_score"] + 1.0) / 2.0
        # desc_ai_n is unbounded but realistic max is ~30
        desc_ai_norm = min(1.0, r["desc_ai_n"] / 30.0)
        # yoe and ai_months are unbounded → log1p them
        yoe_log = math.log1p(max(0.0, r["yoe"]))
        ai_months_log = math.log1p(max(0, r["ai_months"]))
        total_months_log = math.log1p(max(1, r["total_months"]))
        # sbr/sap log
        sbr_log = math.log1p(max(0, r["sbr"]))
        sap_log = math.log1p(max(0, r["sap"]))
        # la_days capped
        la_days_capped = min(r["la_days"], 1000) / 1000.0
        rows.append([
            title_shifted, r["td_match"],
            yoe_log, r["yoe_fit"],
            r["ai_role_pct"], ai_months_log, total_months_log, r["n_skills"],
            r["n_ai_skills"], r["n_ai_adv"], r["n_ai_endorse"],
            desc_ai_norm, r["services_only"], r["loc_score"],
            r["rr"], r["rr_score"],
            la_days_capped, r["la_score"],
            r["np_score"], r["otw"], r["gh_score"], r["pc"], r["v_both"], r["oa_score"],
            sbr_log, sap_log,
            rrf_n[i], bm25_n[i], dense_n[i],
        ])
    X = np.asarray(rows, dtype=np.float32)
    return X, FEATURE_COLS

t0 = time.time()
X, _ = build_feature_matrix(retrieval_records, retrieval_bm25_n, retrieval_dense_n, retrieval_rrf_n)
log.info(f"  ✓ Feature matrix: {X.shape} in {time.time() - t0:.1f}s")

# ----------------------------------------------------------------------------
# 5b. Heuristic silver score — the strongest signals combined with the weights
#     that match the JD's stated priorities. This is the "best-effort" estimate
#     that the LTR head will fit to.
# ----------------------------------------------------------------------------
def heuristic_score(r: dict) -> float:
    """Combine the strongest signals into a single relevance score in [0, 1]."""
    # Title is the dominant signal: trap titles are -1, real AI titles are +1.
    title_s = (r["title_ai_score"] + 1.0) / 2.0  # [0, 1]

    # AI role % is the JD's #1 product fit: "4-5 of 6-8 yrs in applied ML/AI"
    ai_role = r["ai_role_pct"]                     # [0, 1]

    # Description AI mentions → 0..1 via /30
    desc = min(1.0, r["desc_ai_n"] / 30.0)

    # Skills endorsed: only count AI skills that have at least 3 endorsements
    n_ai_endorse = min(1.0, r["n_ai_endorse"] / 5.0)

    # Title-description coherence (catches stitched profiles)
    tdm = r["td_match"]

    # Behavioral
    rr = r["rr_score"]
    la = r["la_score"]
    np_ = r["np_score"]
    otw = r["otw"]
    loc = r["loc_score"]

    # Hard penalties
    svc = r["services_only"]

    # Weighted sum (these weights are deliberate and aligned with the JD)
    score = (
        0.30 * title_s          # title is decisive
      + 0.18 * ai_role          # career in applied ML
      + 0.10 * desc             # AI terms in description
      + 0.08 * n_ai_endorse     # skills actually used & endorsed
      + 0.06 * tdm              # title/description coherence
      + 0.05 * rr               # engagement
      + 0.04 * la               # recency
      + 0.04 * np_              # notice period
      + 0.04 * otw              # open to work
      + 0.03 * loc              # location fit
      - 0.20 * svc              # hard penalty for services-only
    )
    return float(np.clip(score, 0.0, 1.0))

heurs = np.asarray([heuristic_score(r) for r in retrieval_records], dtype=np.float32)
log.info(f"  ✓ Heuristic scores: min={heurs.min():.3f} median={np.median(heurs):.3f} max={heurs.max():.3f}")

# ----------------------------------------------------------------------------
# 5c. Convert heuristic to 5-level silver relevance label.
#     Top 1% → 4 (perfect), top 5% → 3, top 20% → 2, top 50% → 1, rest → 0.
# ----------------------------------------------------------------------------
def to_relevance_labels(scores: np.ndarray) -> np.ndarray:
    """Convert continuous scores to integer labels 0..4 by rank percentile."""
    n = len(scores)
    order = np.argsort(-scores)  # descending
    labels = np.zeros(n, dtype=np.int32)
    pct = np.empty(n, dtype=np.float32)
    for rank, idx in enumerate(order):
        pct[idx] = rank / n
    # thresholds
    labels[pct <= 0.01] = 4
    labels[(pct > 0.01) & (pct <= 0.05)] = 3
    labels[(pct > 0.05) & (pct <= 0.20)] = 2
    labels[(pct > 0.20) & (pct <= 0.50)] = 1
    return labels

y_silver = to_relevance_labels(heurs)
log.info(f"  ✓ Silver label distribution: " +
         ", ".join(f"L{l}={(y_silver==l).sum()}" for l in [4, 3, 2, 1, 0]))

# ----------------------------------------------------------------------------
# 5d. Train XGBoost rank:ndcg
# ----------------------------------------------------------------------------
import xgboost as xgb

t0 = time.time()
group = [len(X)]  # single query = the entire pool

dtrain = xgb.DMatrix(X, label=y_silver)
dtrain.set_group(group)

params = {
    "objective": "rank:ndcg",
    "eval_metric": "ndcg@10",
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "verbosity": 0,
    "seed": SEED,
}

booster = xgb.train(
    params, dtrain,
    num_boost_round=200,
    verbose_eval=False,
)
log.info(f"  ✓ XGBoost rank:ndcg trained in {time.time() - t0:.1f}s")

# Score the full pool
dtest = xgb.DMatrix(X)
ltr_scores = booster.predict(dtest).astype(np.float32)
log.info(f"  ✓ LTR scores: min={ltr_scores.min():.3f} median={np.median(ltr_scores):.3f} max={ltr_scores.max():.3f}")

# ----------------------------------------------------------------------------
# 5e. Feature importance for transparency (printed but not used downstream)
# ----------------------------------------------------------------------------
fi = booster.get_score(importance_type="gain")
fi_sorted = sorted(((FEATURE_COLS[int(k[1:])], v) for k, v in fi.items()), key=lambda x: -x[1])
log.info("  ✓ Top-10 LTR features by gain:")
for fname, gain in fi_sorted[:10]:
    log.info(f"      {fname:24s} {gain:.1f}")

## Phase 6 — Cross-encoder re-rank (top CE_WINDOW) + final selection

We re-rank the top 200 candidates by LTR score with a `ms-marco-MiniLM-L-6-v2` cross-encoder for final precision. The previous notebook ran the cross-encoder on a top-500 window, which is wasteful since most of those candidates are now traps that the pre-filter has removed. With a tight top-200 window, the cross-encoder finishes in <30s on CPU.

In [ ]:
log.info("\n" + "=" * 70)
log.info("PHASE 6 — CROSS-ENCODER RE-RANK (top CE_WINDOW)")
log.info("=" * 70)

# Take the top CE_WINDOW by LTR score from retrieval pool
ce_window = min(CE_WINDOW, len(retrieval_records))
ce_local_idx = np.argsort(-ltr_scores)[:ce_window]
log.info(f"  ✓ Selected top {ce_window} by LTR for cross-encoder")

# Build (jd, candidate) pairs. Truncate candidate text to fit the 512-token
# limit of ms-marco-MiniLM-L-6-v2 (~1800 chars at 4 chars/token).
ce_pairs = []
ce_records = []
for local_i in ce_local_idx:
    r = retrieval_records[local_i]
    text = build_retrieval_text(r)
    text = text[:1800]  # 512 token safety
    ce_pairs.append((jd_ce_text, text))
    ce_records.append((local_i, r))

# Run cross-encoder (CPU; ~30s for 200 pairs)
t0 = time.time()
ce_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
log.info(f"  ✓ Loaded cross-encoder in {time.time() - t0:.1f}s")
t0 = time.time()
ce_scores_raw = ce_model.predict(ce_pairs, show_progress_bar=False)
ce_scores_raw = np.asarray(ce_scores_raw, dtype=np.float32)
log.info(f"  ✓ Cross-encoder scored {len(ce_pairs)} candidates in {time.time() - t0:.1f}s")
log.info(f"     CE range: [{ce_scores_raw.min():.4f}, {ce_scores_raw.max():.4f}]")

# Final order: descending CE score
final_order = np.argsort(-ce_scores_raw)[:TOP_K]
log.info(f"  ✓ Final top {TOP_K} selected.")

# ----------------------------------------------------------------------------
# 6a. Score calibration — we want the final scores to span roughly [0.4, 0.99]
#     so the validator's monotonicity check passes naturally without big jumps.
# ----------------------------------------------------------------------------
ce_top = ce_scores_raw[final_order]
ce_lo, ce_hi = float(ce_top.min()), float(ce_top.max())
if ce_hi - ce_lo < 1e-6:
    ce_norm = np.full(len(final_order), 0.7, dtype=np.float32)
else:
    ce_norm = (ce_top - ce_lo) / (ce_hi - ce_lo)
# Map to [0.55, 0.985] with a slight compression at the bottom
final_scores = 0.55 + 0.435 * ce_norm
# Enforce strict descending order — backward pass
scores_final = np.round(final_scores.astype(np.float64), 4).copy()
for i in range(len(scores_final) - 2, -1, -1):
    if scores_final[i] <= scores_final[i + 1]:
        scores_final[i] = round(scores_final[i + 1] + 0.0001, 4)
# Clamp to [0, 1]
scores_final = np.clip(scores_final, 0.0, 1.0)

log.info(f"  ✓ Score range: [{scores_final.min():.4f}, {scores_final.max():.4f}]")
log.info(f"  ✓ Min gap: {np.diff(scores_final).min():.6f}")

## Phase 7 — Anti-hallucination reasoning

Each `reasoning` field references **only real data** from the candidate's profile. We never invent skills or years. The reasoning format is:

> `<Title> with <YoE> yrs; <one substantive JD-matching reason drawn from real evidence>; <one behavioral signal>`

This passes the Stage 4 hallucination check: every mention is a value the candidate actually has. The substantive reason is chosen from a small set of templates that pull from the candidate's real career evidence.

In [ ]:
log.info("\n" + "=" * 70)
log.info("PHASE 7 — REASONING GENERATION (anti-hallucination)")
log.info("=" * 70)

# ----------------------------------------------------------------------------
# 7a. Build a small library of "evidence phrases" we can pull from the
#     candidate's REAL profile. We never fabricate.
# ----------------------------------------------------------------------------
# Pick the candidate's top-3 AI/ML skills that are in AI_SKILL_CANON.
def real_top_ai_skills(skills: list, n: int = 3) -> list:
    out = []
    for s in skills or []:
        name = s.get("name", "")
        if ai_skill_match(name) and s.get("proficiency") in ("advanced", "expert"):
            out.append(name)
        if len(out) >= n:
            break
    if not out:  # fall back to any AI skill regardless of proficiency
        for s in skills or []:
            if ai_skill_match(s.get("name", "")):
                out.append(s.get("name", ""))
            if len(out) >= n:
                break
    return out[:n]

def real_industry_for_reason(r: dict) -> str:
    """Pick the first product-company industry from career history, if any."""
    for role in r.get("descs", []) or []:
        pass  # we use current_industry here for simplicity
    return (r.get("current_industry") or "").strip()

def location_phrase(r: dict) -> str:
    loc = (r.get("location") or "").strip()
    country = (r.get("country") or "").strip()
    if loc and country:
        return f"{loc}, {country}"
    if country:
        return country
    return "location unspecified"

def build_reasoning(rank: int, r: dict) -> str:
    """Build an anti-hallucination reasoning string for one candidate."""
    title = r.get("title") or "Professional"
    yoe = r.get("yoe", 0.0)

    # Substantive evidence phrase — drawn from real profile fields
    parts = []

    # A) Career evidence
    ai_pct = r.get("ai_role_pct", 0.0)
    if ai_pct >= 0.50:
        years_in_ai = int(round(ai_pct * yoe))
        parts.append(f"{years_in_ai}+ yrs in applied ML/AI roles")
    elif ai_pct >= 0.25:
        parts.append(f"~{int(round(ai_pct * yoe))} yrs in applied ML/AI roles")

    # B) Top AI skills (real names from the profile)
    top_skills = real_top_ai_skills(r.get("skills", []), n=3)
    if top_skills:
        parts.append("skills: " + ", ".join(top_skills))

    # C) Description evidence
    desc_ai_n = r.get("desc_ai_n", 0)
    if desc_ai_n >= 5:
        parts.append(f"{desc_ai_n} AI-domain terms in career description")
    elif desc_ai_n >= 2:
        parts.append("AI-domain language in career history")

    # D) Industry
    ind = real_industry_for_reason(r)
    if ind and ind.lower() in PRODUCT_INDUSTRIES:
        parts.append(f"current industry: {ind}")

    # E) Location fit
    loc = (r.get("location") or "").lower()
    city = loc.split(",")[0].strip()
    if city in INDIA_PREFERRED_CITIES:
        parts.append("Pune/Noida-based (JD preferred)")
    elif city in INDIA_WELCOME_CITIES:
        parts.append(f"{city}-based (JD welcome)")

    # Behavioral phrase
    behav = []
    if r.get("otw", 0) > 0:
        behav.append("open to work")
    la = r.get("la_days", 9999)
    if la <= 30:
        behav.append("active in last 30 days")
    elif la <= 90:
        behav.append("active in last 90 days")
    rr = r.get("rr", 0)
    if rr >= 0.5:
        behav.append(f"recruiter response rate {rr:.2f}")
    np_ = r.get("np_score", 0)
    # We don't have raw notice_period_days in `r` (we stored np_score); pull from saved
    # But the full record isn't here. Re-derive: r doesn't have it directly.
    # Skip the raw days; use a qualitative signal.

    # Title line
    head = f"{title} with {yoe:.1f} yrs"

    body = "; ".join(parts[:3]) if parts else "background aligns with the role"
    tail = ", ".join(behav[:2]) if behav else "standard profile"

    return f"{head}; {body}; {tail}."

reasonings = []
for rank, ord_idx in enumerate(final_order, start=1):
    local_i, r = ce_records[ord_idx]
    reasonings.append(build_reasoning(rank, r))
log.info(f"  ✓ Generated {len(reasonings)} reasonings.")

## Phase 8 — Submission CSV + format validation

Build the DataFrame with columns `candidate_id, rank, score, reasoning`, save with `csv.QUOTE_NONNUMERIC`, and run `validate_submission.py` against the produced CSV. Must print `Submission is valid.`

In [ ]:
log.info("\n" + "=" * 70)
log.info("PHASE 8 — SUBMISSION CSV + FORMAT VALIDATION")
log.info("=" * 70)

# Build the final ordered list of candidate_ids, scores, reasonings
final_ids = []
final_records_for_check = []
for ord_idx in final_order:
    local_i, r = ce_records[ord_idx]
    final_ids.append(r["id"])
    final_records_for_check.append(r)

assert len(final_ids) == TOP_K, f"Expected {TOP_K} ids, got {len(final_ids)}"
assert len(set(final_ids)) == TOP_K, "Duplicate candidate_id in final list!"
assert len(scores_final) == TOP_K
assert len(reasonings) == TOP_K

df_submission = pd.DataFrame({
    "candidate_id": final_ids,
    "rank": list(range(1, TOP_K + 1)),
    "score": scores_final.tolist(),
    "reasoning": reasonings,
})

# Strict descending score check
score_diffs = np.diff(df_submission["score"].values)
assert (score_diffs < 0).all(), "Scores are not strictly descending!"

# Save with proper quoting
df_submission.to_csv(
    OUTPUT_SUBMISSION_PATH,
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
    quotechar='"',
)
log.info(f"  ✓ Saved submission to {OUTPUT_SUBMISSION_PATH}")
log.info(f"     rows: {len(df_submission)}")
log.info(f"     score range: [{df_submission['score'].min():.4f}, {df_submission['score'].max():.4f}]")
log.info(f"     unique candidates: {df_submission['candidate_id'].nunique()}")
log.info("")
log.info("  📋 SUBMISSION PREVIEW (first 10 rows):")
print(df_submission.head(10).to_string(index=False))
log.info("")
log.info("  📋 SUBMISSION PREVIEW (last 5 rows):")
print(df_submission.tail(5).to_string(index=False))

# ----------------------------------------------------------------------------
# 8a. Run the official validator from the challenge bundle
# ----------------------------------------------------------------------------
if os.path.isfile(VALIDATOR_PATH):
    log.info("")
    log.info("  Running validate_submission.py ...")
    import subprocess
    try:
        result = subprocess.run(
            ["python", VALIDATOR_PATH, OUTPUT_SUBMISSION_PATH],
            capture_output=True, text=True, timeout=30
        )
        log.info("  STDOUT: " + (result.stdout.strip() or "(empty)"))
        if result.stderr.strip():
            log.warning("  STDERR: " + result.stderr.strip())
        if result.returncode == 0:
            log.info("  ✅ SUBMISSION IS VALID")
        else:
            log.error(f"  ❌ Validator returned code {result.returncode}")
    except Exception as e:
        log.warning(f"  Could not run validator: {e}")
else:
    log.warning(f"  Skipping validator: {VALIDATOR_PATH} not found.")

## Phase 9 — Self-check report (sanity gate before submit)

This is the manual sanity check the JD asks for. We verify that the top 100 are NOT dominated by trap candidates. If this report shows trap candidates, the submission should not be uploaded.

**Targets (set by the JD's explicit guidance):**
- 0 candidates with `title_ai_score <= -0.5` in top 100 (no trap titles)
- 0 candidates in `SERVICES_COMPANIES` AND `services_only == 1` (services-only)
- 0 honeypots in top 100 (≤10% allowed by spec, we aim for 0)
- ≥80% of top 100 with `title_ai_score >= 0.4` (real AI/ML titles)
- All scores strictly descending

In [ ]:
log.info("\n" + "=" * 70)
log.info("PHASE 9 — SELF-CHECK REPORT (sanity gate)")
log.info("=" * 70)

# Re-derive top-100 summaries
top100 = final_records_for_check

# Trap counts
n_trap_titles = sum(1 for r in top100 if r["title_ai_score"] <= -0.5)
n_services_only = sum(1 for r in top100 if r["services_only"] == 1)
n_honeypots = sum(1 for r in top100 if r["hp"] == 1)
n_strong_pos = sum(1 for r in top100 if r["title_ai_score"] >= 0.7)
n_pos_or_adj = sum(1 for r in top100 if r["title_ai_score"] >= 0.4)

# Behavioral stats
avg_yoe = np.mean([r["yoe"] for r in top100])
avg_ai_pct = np.mean([r["ai_role_pct"] for r in top100])
avg_rr = np.mean([r["rr"] for r in top100])
avg_la_days = np.mean([r["la_days"] for r in top100])
n_india = sum(1 for r in top100 if r.get("country", "").lower() == "india")
n_otw = sum(1 for r in top100 if r["otw"] > 0)

log.info(f"  Top-100 summary:")
log.info(f"    trap titles (score ≤ -0.5):          {n_trap_titles}/100   (target: 0)")
log.info(f"    services-only careers:               {n_services_only}/100   (target: 0)")
log.info(f"    honeypots:                           {n_honeypots}/100   (target: 0)")
log.info(f"    strong-positive titles (≥ +0.7):     {n_strong_pos}/100")
log.info(f"    pos-or-adjacent titles (≥ +0.4):     {n_pos_or_adj}/100  (target: ≥80)")
log.info(f"    India-located:                       {n_india}/100")
log.info(f"    open-to-work:                        {n_otw}/100")
log.info(f"    avg YoE:                             {avg_yoe:.1f} yrs (JD: {JD_YOE_MIN}–{JD_YOE_PREFERRED})")
log.info(f"    avg AI role %:                       {avg_ai_pct:.0%}")
log.info(f"    avg response rate:                   {avg_rr:.2f}")
log.info(f"    avg last-active days:                {avg_la_days:.0f} days")
log.info("")
log.info(f"  Scores:")
log.info(f"    range: [{scores_final.min():.4f}, {scores_final.max():.4f}]")
log.info(f"    gap (rank1 - rank100):               {scores_final[0] - scores_final[-1]:.4f}")
log.info(f"    strictly descending:                 {bool((np.diff(scores_final) < 0).all())}")

# Show the top 10 with title + company
log.info("")
log.info("  TOP-10 CANDIDATES (id, title @ company):")
for i, r in enumerate(top100[:10]):
    log.info(f"    rank {i+1:>3d}: {r['id']}  {r['title']} @ {r['current_company']}  "
             f"(yoe={r['yoe']:.1f}, title_ai={r['title_ai_score']:+.1f}, "
             f"ai_role%={r['ai_role_pct']:.0f}, services={r['services_only']})")

# Pass / fail summary
issues = []
if n_trap_titles > 0:
    issues.append(f"{n_trap_titles} trap titles in top 100")
if n_services_only > 0:
    issues.append(f"{n_services_only} services-only careers in top 100")
if n_honeypots > 10:
    issues.append(f"{n_honeypots} honeypots in top 100 (>10% allowed)")
if not (np.diff(scores_final) < 0).all():
    issues.append("scores not strictly descending")

if not issues:
    log.info("")
    log.info("  ✅ ALL SANITY CHECKS PASSED — safe to submit.")
else:
    log.error("")
    log.error("  ❌ SANITY CHECKS FAILED:")
    for issue in issues:
        log.error(f"     - {issue}")

log.info("")
log.info("=" * 70)
log.info("DONE.")
log.info(f"Submission: {OUTPUT_SUBMISSION_PATH}")
log.info("=" * 70)